# Grid optimization — selection report with protected final test


## Goal
Demonstrate a safe train/validation/test optimization study with multiple metrics, a two-dimensional search space, explicit candidate ranking, untuned-vs-tuned comparison and a protected final test that is never used during hyperparameter selection.


In [ ]:
import os
os.environ.setdefault("MPLBACKEND", "Agg")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DEMO_TEST = os.getenv("MLCORE_DEMO_TEST") == "1"

def balanced_fold_labels(y, n_splits):
    """Demo-only external memberships. Production partition generation belongs to BioSieve."""
    y = np.asarray(y)
    folds = np.empty(len(y), dtype=int)
    for label in np.unique(y):
        idx = np.flatnonzero(y == label)
        folds[idx] = np.arange(len(idx)) % n_splits
    return folds

from sklearn.datasets import make_classification
from mlcore import tune, validate
from mlcore.core import SearchSpace, Categorical
from mlcore.datasets import DatasetBundle, PartitionPlan
from mlcore.tuning import TuningConfig


In [ ]:
n_samples=180 if DEMO_TEST else 360
X,y=make_classification(n_samples=n_samples,n_features=14,n_informative=9,weights=[.62,.38],class_sep=1.0,random_state=31)
ids=np.asarray([f"tune_{i:04d}" for i in range(len(y))],dtype=object)
dataset=DatasetBundle(X=X,y=y,sample_ids=ids,feature_names=[f"f{i}" for i in range(X.shape[1])])
train_ids=[]; val_ids=[]; test_ids=[]
for cls in np.unique(y):
    cls_ids=ids[y==cls]; n=len(cls_ids); a,b=int(.60*n),int(.80*n)
    train_ids.extend(cls_ids[:a]); val_ids.extend(cls_ids[a:b]); test_ids.extend(cls_ids[b:])
plan=PartitionPlan.holdout(train_ids=train_ids,validation_ids=val_ids,test_ids=test_ids,dataset_fingerprint=dataset.fingerprint)


In [ ]:
space=SearchSpace("logreg_grid",{
    "C":Categorical([0.03,0.1,0.3,1.0,3.0]),
    "class_weight":Categorical([None,"balanced"]),
    "solver":Categorical(["lbfgs"]),
})
config=TuningConfig(optimizer="grid",metrics=("mcc","roc_auc","balanced_accuracy"),refit_metric="mcc",random_state=42,n_jobs=1)
opt=tune(dataset=dataset,algorithm="logistic_regression",config=config,partition_plan=plan,search_space=space)
history=opt.history_frame(); complete=history[history.status=="complete"].copy()
score_col="metric__mcc__mean"
top=complete.sort_values(score_col,ascending=False).head(5)
print("Best parameters:",opt.best_params); top[["param__C","param__class_weight",score_col]].head()


In [ ]:
# Baseline and tuned configurations are both evaluated only on the protected test.
final_plan=PartitionPlan.holdout(train_ids=tuple(train_ids)+tuple(val_ids),test_ids=test_ids,
                                 dataset_fingerprint=dataset.fingerprint,name="protected_test")
untuned=validate(dataset=dataset,algorithm="logistic_regression",partition_plan=final_plan,
                 metrics=("mcc","roc_auc","balanced_accuracy"),evaluation_role="test",random_state=42)
tuned=validate(dataset=dataset,algorithm="logistic_regression",partition_plan=final_plan,
               metrics=("mcc","roc_auc","balanced_accuracy"),evaluation_role="test",random_state=42,
               model_params=opt.best_params)
comparison=pd.DataFrame({"untuned":untuned.aggregate_metrics,"tuned":tuned.aggregate_metrics}).T
comparison


In [ ]:
fig,axes=plt.subplots(1,3,figsize=(16,4.5))
for cw,label in [(None,"None"),("balanced","balanced")]:
    part=complete[complete.param__class_weight.astype(str)==str(cw)] if cw is not None else complete[complete.param__class_weight.isna()]
    if not part.empty: axes[0].plot(part.param__C,part[score_col],marker="o",label=label)
axes[0].set_xscale("log"); axes[0].set(title="Grid response",xlabel="C",ylabel="validation MCC"); axes[0].legend()
comparison[["mcc","balanced_accuracy","roc_auc"]].plot.bar(ax=axes[1]); axes[1].set_ylim(-.05,1.05); axes[1].set_title("Protected test")
axes[2].bar(["search","protected test"],[opt.display_scores["mcc"],tuned.aggregate_metrics["mcc"]]); axes[2].set_ylim(-.05,1.05); axes[2].set_title("Selection vs final estimate")
plt.tight_layout(); FIGURE_COUNT=1


In [ ]:
DEMO_CHECKS={
    "test_protected": opt.metadata["protected_samples"]==len(test_ids),
    "ten_candidates": len(complete)==10,
    "multi_metric": set(opt.metrics)=={"mcc","roc_auc","balanced_accuracy"},
    "top_report": len(top)==5,
    "protected_comparison": set(comparison.index)=={"untuned","tuned"},
    "finite_final": np.isfinite(tuned.aggregate_metrics["mcc"]),
    "figure_created": FIGURE_COUNT==1,
}
assert all(DEMO_CHECKS.values()), DEMO_CHECKS
DEMO_CHECKS
